In [ ]:
!pip install sentence-transformers numpy pandas matplotlib scikit-learn

In [ ]:
import json

try:
    with open("code_corpus.json", "r", encoding="utf-8") as f:
        code_corpus = json.load(f)

    with open("eval_questions.json", "r", encoding="utf-8") as f:
        eval_questions = json.load(f)

    print("проверка успешна!")
    print(f"Успешно загружено фрагментов кода: {len(code_corpus)} элементов.")
    print(f"Успешно загружено тестовых вопросов: {len(eval_questions)} элементов.")
except FileNotFoundError as e:
    print(f"Файл не найден")

In [ ]:
from sentence_transformers import SentenceTransformer

models = {
    "MiniLM-L12 (Быстрая)": SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2"),
    "MPNET-Base (Точная)": SentenceTransformer("paraphrase-multilingual-mpnet-base-v2"),
    "E5-Base (Контрастивная)": SentenceTransformer("intfloat/multilingual-e5-base")
}

print("\nвсе модели загружены успешно")

In [ ]:
import numpy as np

import pandas as pd
from sentence_transformers import util

results_table = {}
all_model_embeddings = {}

def prepare_text(chunk, model_name):
    text = f"Функция {chunk['function_name']}: {chunk['description']}"
    return f"passage: {text}" if "E5" in model_name else text

def prepare_query(query_text, model_name):
    return f"query: {query_text}" if "E5" in model_name else query_text

# Запуск вычислений
for model_name, model in models.items():
    print(f"Вычисляем для: {model_name}...")

    corpus_texts = [prepare_text(chunk, model_name) for chunk in code_corpus]
    corpus_embeddings = model.encode(corpus_texts, convert_to_tensor=True)
    all_model_embeddings[model_name] = corpus_embeddings.cpu().numpy()

    correct_top3_count = 0

    for q in eval_questions:
        # Используем точный ключ 'query' из твоего файла
        query_text = prepare_query(q["query"], model_name)
        query_embedding = model.encode(query_text, convert_to_tensor=True)

        cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
        top_results = np.argsort(cos_scores.cpu().numpy())[::-1][:3]

        # Сверяем строковые ID
        top_chunks_ids = [str(code_corpus[idx]["id"]) for idx in top_results]
        correct_id = str(q["correct_chunk_id"])

        if correct_id in top_chunks_ids:
            correct_top3_count += 1

    results_table[model_name] = correct_top3_count / len(eval_questions)

df_results = pd.DataFrame(list(results_table.items()), columns=["Модель", "Precision@3"])
df_results

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np

# беру эмбеддинги лучшей модели (E5-Base)
best_model_key = "E5-Base (Контрастивная)"
best_embeddings = all_model_embeddings[best_model_key]

# извлекаю категории для каждой точки в корпусе
categories = [chunk["category"] for chunk in code_corpus]
unique_categories = sorted(list(set(categories)))

# настройка алгоритма t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(code_corpus)//4))
coords = tsne.fit_transform(best_embeddings)

# график
plt.figure(figsize=(12, 8), dpi=100)

import matplotlib as mpl
colors = mpl.colormaps["tab10"](np.linspace(0, 1, len(unique_categories)))

for i, cat in enumerate(unique_categories):
    idx = [j for j, c in enumerate(categories) if c == cat]
    plt.scatter(coords[idx, 0], coords[idx, 1],
                color=colors[i],
                label=cat,
                alpha=0.85,
                edgecolors='none',
                s=80)

plt.title(f"2D проекция пространства эмбеддингов t-SNE ({best_model_key})", fontsize=14, pad=15)
plt.xlabel("Компонента t-SNE 1", fontsize=11)
plt.ylabel("Компонента t-SNE 2", fontsize=11)

# выношу легенду вбок
plt.legend(title="Категории кода", bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()

# сохраняю график в файл для архива практики
plt.savefig("../Search_Navigator_Nesterova_Violetta/tsne_clusters.png", bbox_inches='tight')
plt.show()